# Setup

In [ ]:
!pip install tqdm

In [ ]:
import os
from ckipnlp.pipeline import CkipPipeline, CkipDocument
pipeline = CkipPipeline(opts={'con_parser': {'username': os.environ["CKIP_USERNAME"], 'password': os.environ["CKIP_PASSWORD"]}})
from os import listdir
import csv
import json
from tqdm import tqdm

In [ ]:
def flatten(l):
    return [item for sublist in l for item in sublist]

In [ ]:
paths=["./files/topics/ntust-news/",
       "./files/topics/nyust-news/",
      ]
tag_titles=['A', 'Caa', 'Cab', 'Cba', 'Cbb', 'D', 'Da', 'Dfa', 'Dfb', 'Di', 'Dk', 'DM', 'I', 'Na', 'Nb', 'Nc', 'Ncd', 'Nd', 'Nep', 'Neqa', 'Neqb', 'Nes', 'Neu', 'Nf', 'Ng', 'Nh', 'Nv', 'P', 'T', 'VA', 'VAC', 'VB', 'VC', 'VCL', 'VD', 'VF', 'VE', 'VG', 'VH', 'VHC', 'VI', 'VJ', 'VK', 'VL', 'V_2', 'DE', 'SHI', 'FW', 'COLONCATEGORY', 'COMMACATEGORY', 'DASHCATEGORY', 'DOTCATEGORY', 'ETCCATEGORY', 'EXCLAMATIONCATEGORY', 'PARENTHESISCATEGORY', 'PAUSECATEGORY', 'PERIODCATEGORY', 'QUESTIONCATEGORY', 'SEMICOLONCATEGORY', 'SPCHANGECATEGORY', 'WHITESPACE']
tag_map={"A":0,"Caa":1,"Cab":2,"Cba":3,"Cbb":4,"D":5,"Da":6,"Dfa":7,"Dfb":8,"Di":9,"Dk":10,"DM":11,"I":12,"Na":13,"Nb":14,"Nc":15,"Ncd":16,"Nd":17,"Nep":18,"Neqa":19,"Neqb":20,"Nes":21,"Neu":22,"Nf":23,"Ng":24,"Nh":25,"Nv":26,"P":27,"T":28,"VA":29,"VAC":30,"VB":31,"VC":32,"VCL":33,"VD":34,"VF":35,"VE":36,"VG":37,"VH":38,"VHC":39,"VI":40,"VJ":41,"VK":42,"VL":43,"V_2":44,"DE":45,"SHI":46,"FW":47,"COLONCATEGORY":48,"COMMACATEGORY":49,"DASHCATEGORY":50,"DOTCATEGORY":51,"ETCCATEGORY":52,"EXCLAMATIONCATEGORY":53,"PARENTHESISCATEGORY":54,"PAUSECATEGORY":55,"PERIODCATEGORY":56,"QUESTIONCATEGORY":57,"SEMICOLONCATEGORY":58,"SPCHANGECATEGORY":59,"WHITESPACE":60,}
fn="./files/is-topic_trainning_set/false.csv"
topics=[]
for path in paths:
    for file in listdir(path):
        topics.append(path + file) 

# Testing Tool

#https://www.tutorialspoint.com/jupyter/jupyter_notebook_markdown_cells.htm

# tqdm example:
for i in tqdm(range(10)):
   somthing...
ref:https://stackoverflow.com/questions/3160699/python-progress-bar

# Multiprocess Functions

In [ ]:
import multiprocessing as mp
import time
from os.path import exists

## Listening for messages on the quere(q), writes to file.
To avoid collisions, only listener function writting message to file

In [ ]:
def listener(q):
    #if file not exists, create a file and add header
    if not exists(fn):
        with open(fn, 'w') as f:
            writer = csv.writer(f)
            writer.writerow(tag_titles)   
    #listening to quere       
    with open(fn, 'a') as f:
        writer = csv.writer(f)
        while 1:
            m = q.get()
            if m == 'shutdown':
                break
            writer.writerow(m)
            f.flush()

## Caculate ckipnlp tagger result

In [ ]:
def worker(path, q):
    with open(path, mode="r") as f:       
        jsonn = json.loads(f.read())
        tags=[]
        for info in jsonn["content"]:
            doc = CkipDocument(raw = info)
            pipeline.get_pos(doc)
            tags += flatten(doc.pos.to_list())
        word_counter = 0
        #make a zeros array, use numpy will be better.
        tag_counter = [0]*len(tag_map)
        for tag in tags:
            word_counter += 1
            tag_counter[tag_map[tag]] += 1
        if(word_counter > 10):# document must include more than 10 words
            for t in range(len(tag_counter)):
                # 0 ~ 1
                tag_counter[t] = format(tag_counter[t] / word_counter, '.6f')
            q.put(tag_counter)

In [ ]:
def main():
    #must use Manager queue here, or will not work.
    manager = mp.Manager()
    q = manager.Queue()
    # "mp.cpu_count() + 2" Common practice to ensure that the pool will be saturated.
    # Each kip tagger process use about 2.5G ram, make sure is meomory enough.
    pool = mp.Pool(16)

    #put listener to work first
    watcher = pool.apply_async(listener, (q,))
    
    progress = tqdm(total=len(topics))
    #fire off workers
    jobs = []
    for i in topics:
        job = pool.apply_async(worker, (i, q))
        jobs.append(job)

    # collect results from the workers through the pool result queue
    for job in jobs:
        job.get()   
        progress.update(1)

    #now we are done, kill the listener
    q.put('shutdown')
    pool.close()
    pool.join()

In [ ]:
if __name__ == "__main__":
   main()